# Random Split Model Evaluation

This experiment trains on a random 70% of route rows drawn from all eight weeks, compares models on a 15% validation sample, and reports final metrics on a separate 15% test sample. Seed 42 makes the split reproducible.

This answers how well the model predicts another route from the same season-wide mix. It complements the earlier week-based split, which asks whether a model trained on earlier weeks carries forward to a later week.

In [1]:
from pathlib import Path
import csv
import sys
from collections import Counter

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from separation_over_expected.reports import read_csv_rows

ROUTE_TABLE = PROJECT_ROOT / "data" / "processed" / "route_level_snap_to_release.csv"
RANDOM_DIR = PROJECT_ROOT / "data" / "processed" / "random_split" / "position_baselines"
WEEK_DIR = PROJECT_ROOT / "data" / "processed" / "position_baselines"

rows = read_csv_rows(RANDOM_DIR / "route_level_baseline_predictions_all.csv")
len(rows)

35032

## Split Coverage

Each partition contains routes from all eight weeks. A route is the sampling unit, so routes from the same play can land in different partitions. Notebook 06 follows up with a game-grouped split to check the effect of keeping games together.

In [2]:
split_counts = Counter(row["split"] for row in rows)
week_counts = {
    split: dict(sorted(Counter(row["week"] for row in rows if row["split"] == split).items()))
    for split in ("train", "validation", "test")
}
{"split_counts": dict(split_counts), "routes_by_week": week_counts}

{'split_counts': {'train': 24522, 'validation': 5254, 'test': 5256}, 'routes_by_week': {'train': {'1': 3472, '2': 3047, '3': 3298, '4': 3127, '5': 3126, '6': 2883, '7': 2636, '8': 2933}, 'validation': {'1': 726, '2': 640, '3': 684, '4': 693, '5': 708, '6': 629, '7': 506, '8': 668}, 'test': {'1': 729, '2': 632, '3': 675, '4': 654, '5': 711, '6': 650, '7': 554, '8': 651}}}

## Test Metrics: All Models and Positions

The table compares the original week-based evaluation with the new season-wide random evaluation. Both use the same model definitions and tracked route table. Random split receiver summaries are computed from test routes only.

In [3]:
positions = [("All", "all"), ("WR", "wr"), ("TE", "te"), ("RB", "rb")]
comparison = []
for position, suffix in positions:
    for strategy, directory in (("Week holdout", WEEK_DIR), ("Random all weeks", RANDOM_DIR)):
        metric_rows = read_csv_rows(directory / f"baseline_metrics_{suffix}.csv")
        for row in metric_rows:
            if row["split"] == "test":
                comparison.append({
                    "position": position,
                    "evaluation": strategy,
                    "model": row["model"],
                    "routes": row["n"],
                    "r2": row["r2"],
                    "rmse": row["rmse"],
                    "mae": row["mae"],
                })
comparison

[{'position': 'All', 'evaluation': 'Week holdout', 'model': 'global_mean', 'routes': '4252', 'r2': '-0.003', 'rmse': '2.904', 'mae': '2.198'}, {'position': 'All', 'evaluation': 'Week holdout', 'model': 'smoothed_group_mean', 'routes': '4252', 'r2': '0.045', 'rmse': '2.834', 'mae': '2.142'}, {'position': 'All', 'evaluation': 'Week holdout', 'model': 'ridge_context', 'routes': '4252', 'r2': '0.331', 'rmse': '2.372', 'mae': '1.720'}, {'position': 'All', 'evaluation': 'Random all weeks', 'model': 'global_mean', 'routes': '5256', 'r2': '-0.000', 'rmse': '2.953', 'mae': '2.236'}, {'position': 'All', 'evaluation': 'Random all weeks', 'model': 'smoothed_group_mean', 'routes': '5256', 'r2': '0.050', 'rmse': '2.878', 'mae': '2.187'}, {'position': 'All', 'evaluation': 'Random all weeks', 'model': 'ridge_context', 'routes': '5256', 'r2': '0.359', 'rmse': '2.363', 'mae': '1.711'}, {'position': 'WR', 'evaluation': 'Week holdout', 'model': 'global_mean', 'routes': '2474', 'r2': '-0.004', 'rmse': '2.6

## Wide Receiver Test-Set Summary

The WR ridge model is the main project surface. These receiver-level summaries use only held-out test routes, with at least 25 routes in the test partition.

In [4]:
wr_summary = read_csv_rows(RANDOM_DIR / "receiver_baseline_summary_wr.csv")
wr_summary[:15]

[{'nflId': '52442', 'displayName': 'Michael Pittman', 'officialPosition': 'WR', 'routes': '25', 'mean_soe': '1.382', 'median_soe': '0.886', 'std_soe': '2.664', 'se_soe': '0.533', 'lower_95_soe': '0.337', 'upper_95_soe': '2.426', 'total_soe': '34.541'}, {'nflId': '48374', 'displayName': 'Olamide Zaccheaus', 'officialPosition': 'WR', 'routes': '27', 'mean_soe': '0.487', 'median_soe': '-0.062', 'std_soe': '1.958', 'se_soe': '0.377', 'lower_95_soe': '-0.252', 'upper_95_soe': '1.225', 'total_soe': '13.137'}, {'nflId': '40024', 'displayName': 'Keenan Allen', 'officialPosition': 'WR', 'routes': '36', 'mean_soe': '0.178', 'median_soe': '0.032', 'std_soe': '1.328', 'se_soe': '0.221', 'lower_95_soe': '-0.256', 'upper_95_soe': '0.612', 'total_soe': '6.410'}, {'nflId': '43808', 'displayName': 'Robby Anderson', 'officialPosition': 'WR', 'routes': '39', 'mean_soe': '0.210', 'median_soe': '-0.055', 'std_soe': '1.540', 'se_soe': '0.247', 'lower_95_soe': '-0.273', 'upper_95_soe': '0.693', 'total_soe': 

## Interpretation

The random split evaluates interpolation across the observed season-wide distribution. The week holdout evaluates later-week transfer. Similar results would suggest the model is not highly sensitive to that particular week boundary; differences can reflect distribution shift as well as test-sample variation.

This experiment uses a route-level random split. It does not establish that routes within a play are statistically independent, and the test results should be described as same-season route prediction rather than future-season performance. See notebook 06 for the grouped-game robustness check.